In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [4]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Llama-3.1-70B-bnb-4bit",
    "unsloth/Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.18: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.3.18 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [6]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples, for_training=True):
    texts = []
    for instruction, input_text, label in zip(
        examples["instruction"], examples["input"], examples["output"]
    ):
        if for_training:
            text = alpaca_prompt.format(
                instruction,
                input_text,
                label
            ) + EOS_TOKEN
        else:  # inference
            text = alpaca_prompt.format(
                instruction,
                input_text,
                ""
            )
        texts.append(text)
    return {"text": texts}

import json
from datasets import Dataset
from sklearn.model_selection import train_test_split

# Load file
with open("/content/ML_relations_fixed.json", "r") as f:
    data = json.load(f)

# Split
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Convert to HF datasets
train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

# Apply formatting
train_dataset = train_dataset.map(
    lambda x: formatting_prompts_func(x, for_training=True),
    batched=True
)

test_dataset = test_dataset.map(
    lambda x: formatting_prompts_func(x, for_training=False),
    batched=True
)

Map:   0%|          | 0/1569 [00:00<?, ? examples/s]

Map:   0%|          | 0/393 [00:00<?, ? examples/s]

In [7]:
from trl import SFTConfig, SFTTrainer

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,   # ✅ CORRECT
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        num_train_epochs = 2,
        learning_rate = 1e-5,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Map:   0%|          | 0/1569 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1569 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [8]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
6.707 GB of memory reserved.


In [9]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,569 | Num Epochs = 2 | Total steps = 394
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,3.232700
2,3.195100
3,3.279100
4,3.399400
5,3.302900
6,3.216100
7,3.288300
8,3.093200
9,3.181100
10,3.240600


Unsloth: Will smartly offload gradients to save VRAM!


In [15]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import re

LABELS = [
    "used_for",
    "based_on",
    "implements",
    "part_of",
    "improves",
    "no_relation"
]
def clean_prediction(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9_ ]", "", text)

    for label in LABELS:
        if label in text:
            return label

    return "no_relation"  # fallback

def predict(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=10,
        temperature=0.0,
        do_sample=False
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only response part
    if "### Response:" in decoded:
        decoded = decoded.split("### Response:")[-1].strip()

    return decoded

y_true = []
y_pred = []

for item in test_data:
    prompt = f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{item['instruction']}

### Input:
{item['input']}

### Response:
"""

    pred = predict(model, tokenizer, prompt)

    y_pred.append(clean_prediction(pred))
    y_true.append(clean_prediction(item["output"]))

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted")
recall = recall_score(y_true, y_pred, average="weighted")
f1 = f1_score(y_true, y_pred, average="weighted")

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy: 0.539440203562341
Precision: 0.532426107950917
Recall: 0.539440203562341
F1 Score: 0.476111555819732


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [17]:

LABELS = [
    "used_for",
    "based_on",
    "implements",
    "part_of",
    "improves",
    "no_relation"
]
def clean_prediction(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9_ ]", "", text)

    for label in LABELS:
        if label in text:
            return label

    return "no_relation"  # fallback


import re
import json

def extract_entities_and_types(text):
    try:
        e1_match = re.search(r"Entity1:\s*(.*?)\s*\((.*?)\)", text)
        e2_match = re.search(r"Entity2:\s*(.*?)\s*\((.*?)\)", text)

        e1 = e1_match.group(1).strip()
        e1_type = e1_match.group(2).strip()

        e2 = e2_match.group(1).strip()
        e2_type = e2_match.group(2).strip()

        return e1, e1_type, e2, e2_type

    except:
        return "UNKNOWN", "OTHER", "UNKNOWN", "OTHER"


# ✅ Build triplets
triplets = []

for item, pred in zip(test_data, y_pred):

    input_text = item["input"]

    relation = clean_prediction(pred)

    e1, e1_type, e2, e2_type = extract_entities_and_types(input_text)

    triplets.append({
        "entity1": e1,
        "entity1_type": e1_type,
        "relation": relation,
        "entity2": e2,
        "entity2_type": e2_type
    })


# ✅ SAVE FILE (JSON format)
output_path = "/content/llama_predicted_triplets.json"

with open(output_path, "w") as f:
    json.dump(triplets, f, indent=4)

print(f"✅ Triplets saved to: {output_path}")

✅ Triplets saved to: /content/llama_predicted_triplets.json
